# STT Lab

Research workflow:

1. Browse catalog / runnable models
2. Compare on your audio
3. Pick a base model
4. Build a personal dataset
5. Fine-tune locally (or cloud when configured)
6. Evaluate before / after

**Kernel:** STT Lab

## 1. Setup

In [ ]:
from pathlib import Path
import sys

NB_DIR = Path.cwd()
if (NB_DIR / "helpers.py").exists():
    ROOT = NB_DIR.parent
elif (NB_DIR / "notebooks" / "helpers.py").exists():
    ROOT = NB_DIR
    sys.path.insert(0, str(NB_DIR / "notebooks"))
else:
    raise RuntimeError("Run from stt-lab/ or stt-lab/notebooks/")

import helpers as h
print("Root:", ROOT)
h.models_df()

## 2. Research — catalog

Filter candidates before wiring or comparing.

In [ ]:
# What we can add next vs what is already runnable
wired = h.catalog_df(status="wired")[["id", "name", "mode", "roles"]]
easy_adapt = h.catalog_df(status="easy", role="adapt")[["id", "name", "family", "finetune"]].head(15)
print("wired:\n", wired.to_string(index=False))
print("\neasy adapt candidates:\n", easy_adapt.to_string(index=False))
print("\npriority adds:", h.load_catalog()["experiment_matrices"]["priority_adds"][:8])

## 3. Compare — pick a model

Use a real voice clip + reference transcript.

In [ ]:
AUDIO = ROOT / "data" / "audio" / "your-voice.wav"  # change me
REFERENCE = "paste the exact words you said"
MODEL_IDS = ["whisper-tiny", "whisper-base"]  # pick from h.models_df()

assert AUDIO.exists(), f"Missing audio: {AUDIO}"

resp = h.compare(AUDIO, MODEL_IDS, reference=REFERENCE or None)
df = h.results_df(resp)
df

In [ ]:
for r in resp.results:
    print(f"=== {r.model_name} ===")
    if r.error:
        print("ERROR:", r.error)
    else:
        print("transcript:", r.transcript)
        print("diff:     ", h.show_diff(REFERENCE, r.transcript))
    print()

# Choose the base model you will fine-tune
CHOSEN_BASE = "tiny"  # tiny | base | small | medium

## 4. Personal dataset

Need train + val before fine-tune.

In [ ]:
dataset_id = h.create_dataset("my voice set")
h.add_sample(dataset_id, AUDIO, REFERENCE, split="train")
h.add_sample(dataset_id, AUDIO, REFERENCE, split="val")
h.list_datasets()

## 5. Fine-tune

`backend="local"` runs Whisper LoRA here.  
`backend="cloud"` is a reserved hook (Modal / HF Jobs / RunPod) — not implemented yet.

In [ ]:
BACKEND = "local"  # or "cloud" when configured

job_id = h.start_finetune(
    dataset_id,
    base_model=CHOSEN_BASE,
    epochs=1,
    lora_rank=8,
    backend=BACKEND,
)
print("job_id:", job_id)
status = h.wait_for_job(job_id)
status

## 6. Evaluate before / after

In [ ]:
if status["status"] != "completed":
    raise RuntimeError(f"Fine-tune did not complete: {status}")

ev = h.evaluate(dataset_id, base_model=CHOSEN_BASE, adapter_id=job_id, split="val")
print(f"base WER={ev.base_wer}  adapted WER={ev.adapted_wer}  Δ={ev.delta_wer}")
h.eval_df(ev)

## 7. Re-compare with adapted model

In [ ]:
adapted = h.models_df()
adapted = adapted[adapted["provider"] == "adapted"]
print(adapted.to_string(index=False) if len(adapted) else "No adapted models yet")

ids = [f"whisper-{CHOSEN_BASE}"]
if len(adapted):
    ids.append(adapted.iloc[0]["id"])

h.results_df(h.compare(AUDIO, ids, reference=REFERENCE or None))